In [ ]:
import os
import random
import shutil

# Configuración
root_dir = 'yolo_dataset'  # Donde están tus carpetas 'images/train' y 'labels/train'
output_dir = 'final_dataset'

# Proporciones (deben sumar 1.0)
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

# 1. Crear estructura de carpetas
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

# 2. Obtener lista de archivos (basándonos en las imágenes)
img_source = os.path.join(root_dir, 'images')
lbl_source = os.path.join(root_dir, 'labels')

all_images = [f for f in os.listdir(img_source) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))]
random.shuffle(all_images)

# 3. Calcular índices de corte
total = len(all_images)
train_idx = int(total * train_ratio)
val_idx = int(total * (train_ratio + val_ratio))

# 4. Función para mover archivos
def move_files(files, split):
    for f in files:
        # Nombre base sin extensión para encontrar el .txt
        base_name = os.path.splitext(f)[0]
        
        # Mover Imagen
        shutil.copy(os.path.join(img_source, f), 
                    os.path.join(output_dir, 'images', split, f))
        
        # Mover Label (.txt)
        label_file = f"{base_name}.txt"
        if os.path.exists(os.path.join(lbl_source, label_file)):
            shutil.copy(os.path.join(lbl_source, label_file), 
                        os.path.join(output_dir, 'labels', split, label_file))

# 5. Ejecutar el reparto
move_files(all_images[:train_idx], 'train')
move_files(all_images[train_idx:val_idx], 'val')
move_files(all_images[val_idx:], 'test')

print(f"Dataset dividido con éxito en '{output_dir}':")
print(f"- Train: {train_idx} imágenes")
print(f"- Val: {val_idx - train_idx} imágenes")
print(f"- Test: {total - val_idx} imágenes")

Error: No se encuentra la carpeta images


In [1]:
from ultralytics import YOLO
import torch

# 1. Verificación de Hardware
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🚀 Entrenando en: {torch.cuda.get_device_name(0) if device == 0 else 'CPU'}")

# 2. Cargar el modelo YOLO11n (la versión más actual y veloz)
# Si prefieres la versión v8, puedes usar 'yolov8n.pt'
model = YOLO('yolo11n.pt') 

# 3. Iniciar Entrenamiento
results = model.train(
    data='data_images.yaml',       # Tu archivo de configuración del dataset
    epochs=300,             # Un límite alto, la paciencia se encargará del resto
    project='.',
    imgsz=640,              # Tamaño de imagen estándar
    batch=-1,               # Auto-ajuste de batch según tu VRAM (RTX 3050)
    device=device,          # Usa tu GPU NVIDIA
    patience=50,            # Si el modelo no mejora en 50 épocas, se detiene solo
    
    # --- AUMENTO DE IMÁGENES (Augmentation) ---
    mosaic=1.0,             # Mezcla 4 imágenes para ver objetos a distintas escalas
    mixup=0.1,              # Superpone imágenes para mejorar la robustez
    hsv_h=0.015,            # Ajuste aleatorio de tono
    hsv_s=0.7,              # Ajuste de saturación
    hsv_v=0.4,              # Ajuste de brillo
    degrees=10.0,           # Rotaciones aleatorias de hasta 10 grados
    fliplr=0.5,             # Volteo horizontal aleatorio (50% de probabilidad)
    
    name='yolo_model_final', # Nombre de la carpeta de salida
    save=True               # Guarda los checkpoints automáticamente
)

print("✅ Entrenamiento finalizado. Los mejores pesos están en: runs/detect/yolo_model_final/weights/best.pt")

🚀 Entrenando en: NVIDIA RTX 6000 Ada Generation
Ultralytics 8.4.22 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_images.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_model_final, nbs=64, nms=False, opset=None, optimize=

# ANalisis metricas


El modelo presenta un rendimiento solido con un mAP50 global de 0.866, lo cual es excelente para un despliegue en tiempo real. Con un tiempo de inferencia de apenas 0.3ms por imagen, el sistema es extremadamente ligero y capaz de procesar cientos de cuadros por segundo en hardware adecuado.

# DESEMPEÑO POR CLASE

    Headphone: Es la clase con mejor desempeño. Su Precision (0.984) y Recall (0.846) indican que el modelo casi nunca se equivoca al identificar unos audifonos y es capaz de encontrar la gran mayoria de los presentes en la escena. El mAP50 de 0.919 confirma que la deteccion es muy robusta.

    Credential: Presenta un rendimiento ligeramente inferior con un mAP50 de 0.813. El Recall de 0.75 sugiere que en un 25% de los casos el modelo podria no detectar la credencial, probablemente debido a variaciones de angulo, reflejos en el plastico o tamaño pequeño en la imagen.

# FORTALEZAS

    Alta precision general (0.899): El sistema genera muy pocos falsos positivos, lo que evita alertas innecesarias por objetos que no son credenciales ni audifonos.

    Localizacion precisa: Un mAP50-95 de 0.635 es un valor alto que indica que los cuadros delimitadores (bounding boxes) se ajustan con gran exactitud a los bordes de los objetos.

# AREAS DE MEJORA

    El Recall global de 0.798 es el punto mas debil. Para mejorar el control de acceso en el laboratorio, se recomienda aumentar el dataset de la clase 'credential' incluyendo imagenes con diferentes condiciones de luz y orientaciones para reducir las omisiones de deteccion.

In [2]:
from ultralytics import YOLO
import cv2

model_path = 'runs/detect/yolo_model_final/weights/best.pt'
model = YOLO(model_path)

# 2. Inicializar la cámara (0 suele ser la webcam integrada)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: No se pudo acceder a la cámara.")
    exit()

print("🚀 Iniciando detección... Presiona 'q' para salir.")

while True:
    # Leer un cuadro de la cámara
    success, frame = cap.read()
    
    if not success:
        break

    # 3. Inferencia con YOLO
    # stream=True es más eficiente para video
    # conf=0.5 para evitar falsos positivos
    results = model.predict(source=frame, conf=0.6, device=0, stream=True)

    # 4. Dibujar los resultados en el cuadro
    for r in results:
        annotated_frame = r.plot()

        # 5. Mostrar el video con las detecciones
        cv2.imshow("YOLO11 Real-Time Detection", annotated_frame)

    # Romper el bucle si se presiona la tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Liberar recursos
cap.release()
cv2.destroyAllWindows()

🚀 Iniciando detección... Presiona 'q' para salir.

0: 480x640 (no detections), 10.3ms
Speed: 1.6ms preprocess, 10.3ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 11.0ms
Speed: 1.1ms preprocess, 11.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 13.4ms
Speed: 1.3ms preprocess, 13.4ms inference, 0.8ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.5ms
Speed: 1.3ms preprocess, 9.5ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.4ms
Speed: 1.2ms preprocess, 9.4ms inference, 0.5ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.4ms
Speed: 1.1ms preprocess, 9.4ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 9.2ms
Speed: 1.0ms preprocess, 9.2ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections)